<a href="https://colab.research.google.com/github/AnjiLakshmi12/6thSem-ML-Lab/blob/main/1BM24CS401_Lab_7_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import csv
import random
from collections import Counter
import os
import pandas as pd
from sklearn.datasets import load_iris

# --- Start of Iris CSV creation ---
# Create the directory if it doesn't exist
output_dir = '/content/sample_data/'
os.makedirs(output_dir, exist_ok=True)

# Load the Iris dataset
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
# Add the 'target' column first
iris_df['target'] = iris.target
# Add species names for target column, as the original example used string labels
iris_df['species'] = iris_df['target'].apply(lambda x: iris.target_names[x])

# We need features and target, similar to the original dataset format
# Original format was ([float_feature1, float_feature2, ...], 'label_string')
# Let's adjust for that by dropping the numerical target column and reordering
iris_output_df = iris_df[iris.feature_names + ['species']]

# Save the DataFrame to a CSV file without header
csv_path = os.path.join(output_dir, 'iris.csv')
iris_output_df.to_csv(csv_path, index=False, header=False) # Important: no header

print(f"'iris.csv' created at {csv_path}")
# --- End of Iris CSV creation ---


def load_data(filename):
    data = []
    with open(filename) as f:
        reader = csv.reader(f)
        # next(reader) # Removed: The CSV is now created without a header
        for row in reader:
            # Ensure row has enough elements before accessing row[:-1] and row[-1]
            if len(row) > 1:
                data.append(([float(x) for x in row[:-1]], row[-1]))
            else:
                print(f"Skipping malformed row: {row}") # Debugging aid
    return data

# Split Dataset

def train_test_split(dataset, test_size=0.2):
    random.shuffle(dataset)
    split = int(len(dataset) * (1 - test_size))
    return dataset[:split], dataset[split:]

# Gini Index

def gini(groups, classes):
    total = sum(len(group) for group in groups)
    gini_score = 0.0

    for group in groups:
        size = len(group)
        if size == 0:
            continue
        score = 0.0
        labels = [row[1] for row in group]
        for class_val in classes:
            p = labels.count(class_val) / size
            score += p * p
        gini_score += (1 - score) * (size / total)

    return gini_score

# Split Dataset based on feature

def split(index, value, dataset):
    left, right = [], []
    for row in dataset:
        if row[0][index] < value:
            left.append(row)
        else:
            right.append(row)
    return left, right

# Get Best Split

def get_best_split(dataset, n_features):
    class_values = list(set(row[1] for row in dataset))
    best_index, best_value, best_score, best_groups = None, None, float("inf"), None

    # Cap n_features to the actual number of available features
    num_actual_features = len(dataset[0][0]) if dataset else 0
    # Ensure n_features is at least 1 if dataset has features, otherwise 0
    n_features_to_sample = min(n_features, num_actual_features) if num_actual_features > 0 else 0

    # Handle case where n_features_to_sample might be 0, to avoid random.sample error
    if n_features_to_sample == 0:
        return {'index': None, 'value': None, 'groups': ([], [])}

    features = random.sample(range(num_actual_features), n_features_to_sample)

    for index in features:
        for row in dataset:
            groups = split(index, row[0][index], dataset)
            gini_score = gini(groups, class_values)
            if gini_score < best_score:
                best_index, best_value, best_score, best_groups = index, row[0][index], gini_score, groups

    return {'index': best_index, 'value': best_value, 'groups': best_groups}

# Create Leaf Node

def to_leaf(group):
    outcomes = [row[1] for row in group]
    if not outcomes: # Handle empty group case
        return None # Or some default value if applicable
    return Counter(outcomes).most_common(1)[0][0]

# Build Tree

def build_tree(dataset, max_depth, min_size, n_features, depth=0):
    # Handle empty dataset for recursive calls or initial call
    if not dataset:
        return None

    split_info = get_best_split(dataset, n_features)
    index, value, groups = split_info['index'], split_info['value'], split_info['groups']
    left, right = groups

    # If no valid split found or groups are empty after split, create a leaf
    if index is None or not left or not right:
        return to_leaf(dataset)

    # Check for max depth or minimum size for a split, if so, create a leaf
    if depth >= max_depth or len(left) < min_size or len(right) < min_size:
        return to_leaf(left + right) # Make combined groups a leaf

    node = {'index': index, 'value': value}
    node['left'] = build_tree(left, max_depth, min_size, n_features, depth + 1)
    node['right'] = build_tree(right, max_depth, min_size, n_features, depth + 1)

    # Handle cases where recursive calls might return None due to empty datasets
    if node['left'] is None: # This could happen if a sub-split leads to an empty dataset
        node['left'] = to_leaf(left) if left else None # Create a leaf from 'left' if possible, else None
    if node['right'] is None:
        node['right'] = to_leaf(right) if right else None

    return node

# Predict with Tree

def predict_tree(node, row):
    if isinstance(node, dict):
        # Access index and value directly as build_tree now ensures they exist
        if row[node['index']] < node['value']:
            return predict_tree(node['left'], row)
        else:
            return predict_tree(node['right'], row)
    else: # This is a leaf node
        return node

# Bootstrap Sampling

def bootstrap_sample(dataset):
    sample = []
    n = len(dataset)
    if n == 0: # Handle empty dataset case
        return []
    for _ in range(n):
        sample.append(random.choice(dataset))
    return sample

# Random Forest

def random_forest(train, test, n_trees, max_depth, min_size, n_features):
    trees = []

    for _ in range(n_trees):
        sample = bootstrap_sample(train)
        if not sample: # Skip if bootstrap sample is empty
            continue
        tree = build_tree(sample, max_depth, min_size, n_features)
        if tree is not None: # Only add valid trees
            trees.append(tree)

    if not trees: # If no trees were built, return empty predictions
        return []

    predictions = []
    for row in test:
        votes = [predict_tree(tree, row[0]) for tree in trees if tree is not None]
        if votes: # Only get majority vote if there are actual votes
            predictions.append(Counter(votes).most_common(1)[0][0])
        else:
            # Fallback if no trees or no votes for a given row
            # Could be a random guess or a default class, for now, let's skip or assign a placeholder
            pass # Or predictions.append(most_common_class_in_train_data)

    return predictions

# Accuracy

def accuracy(actual, predicted):
    if not actual or not predicted:
        return 0.0
    if not actual and not predicted: # Both empty, consider 100% accurate if no data to predict
        return 100.0
    correct = sum(1 for i in range(len(actual)) if actual[i] == predicted[i])
    return correct / len(actual) * 100

# Main Execution

dataset = load_data(csv_path) # Use the dynamically created csv_path
train, test = train_test_split(dataset)

actual = [row[1] for row in test]

# Default RF (10 trees)
predictions = random_forest(train, test, n_trees=10, max_depth=5, min_size=2, n_features=2)
acc = accuracy(actual, predictions)

print("Accuracy with 10 trees:", acc)

# Fine tuning
best_acc = 0
best_trees = 0

for trees in [5, 10, 20, 50]:
    preds = random_forest(train, test, trees, max_depth=5, min_size=2, n_features=2)
    acc = accuracy(actual, preds)

    print(f"Trees: {trees}, Accuracy: {acc})")

    if acc > best_acc:
        best_acc = acc
        best_trees = trees

print("\nBest Accuracy:", best_acc)
print("Best Number of Trees:", best_trees)


'iris.csv' created at /content/sample_data/iris.csv
Accuracy with 10 trees: 93.33333333333333
Trees: 5, Accuracy: 93.33333333333333)
Trees: 10, Accuracy: 86.66666666666667)
Trees: 20, Accuracy: 93.33333333333333)
Trees: 50, Accuracy: 93.33333333333333)

Best Accuracy: 93.33333333333333
Best Number of Trees: 5
